# Dairy vs Beef Cattle: Differential Heat Stress Responses and Mortality

## Scientific Background

Dairy and beef cattle exhibit **fundamentally different physiological responses** to heat stress due to:

### Physiological Differences

**Dairy Cattle (Higher Vulnerability):**
- **Higher metabolic heat production**: Lactation increases heat load by 20-30%
- **Greater feed intake**: More metabolic heat from digestion
- **Selective breeding**: Emphasis on milk production over heat tolerance
- **Body composition**: Less subcutaneous fat for insulation but more metabolic tissue
- **Behavioral constraints**: Milking schedules limit shade-seeking behavior

**Beef Cattle (Moderate Vulnerability):**
- **Lower metabolic rate**: No lactation burden
- **Natural selection**: Many breeds retain heat tolerance traits
- **Behavioral flexibility**: Can seek shade during peak heat
- **Feed efficiency**: Lower intake reduces digestive heat
- **Coat color variations**: Some breeds have reflective coats

### Critical Temperature Thresholds

- **Dairy**: Thermoneutral zone 5-25°C; heat stress begins at 25-27°C
- **Beef**: Thermoneutral zone 5-30°C; heat stress begins at 30-32°C
- **Lactating dairy**: Even lower threshold (~22°C with high humidity)

### Research Questions

1. Do dairy cattle show stronger mortality responses to heat stress than beef?
2. Are threshold temperatures different between cattle types?
3. Does VPD affect dairy more than beef cattle?
4. Do seasonal patterns differ between dairy and beef mortality?
5. Are regional differences consistent across cattle types?

### Hypotheses

**H1**: Dairy mortality correlates more strongly with heat metrics than beef  
**H2**: Dairy cattle show mortality increases at lower temperature thresholds  
**H3**: VPD effects are stronger for dairy (impaired evaporative cooling critical)  
**H4**: Summer dairy mortality spikes are higher than beef  
**H5**: Regional patterns differ: dairy more vulnerable in humid Southeast, beef in arid Southwest

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Import project configuration
import sys
sys.path.append('../../')
from config import (
    STATE_NAMES, STATE_ABBRS, STATE_REGIONS, 
    FOCUS_STATES, CATTLE_REGIONS, CUSTOM_REGIONS, SEASONS
)

# Set plotting style
sns.set_style('whitegrid')
sns.set_palette('Set1')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully")
print(f"Focus states (n={len(FOCUS_STATES)}): {sorted([STATE_ABBRS[s] for s in FOCUS_STATES.values()])}")

## 1. Data Loading and Preparation

In [ ]:
# Load merged cattle-heat dataset
cattle_heat = pd.read_csv('../../cattle_data/cattle_heat_merged.csv', parse_dates=['week_ending'])
print(f"Loaded {len(cattle_heat)} weeks of data ({cattle_heat['week_ending'].min()} to {cattle_heat['week_ending'].max()})")

# Check available cattle type columns
print(f"\nAvailable columns:")
cattle_cols = [col for col in cattle_heat.columns if 'dairy' in col.lower() or 'beef' in col.lower() or 'total' in col.lower()]
print(cattle_cols)

# Add temporal features
cattle_heat['year'] = cattle_heat['week_ending'].dt.year
cattle_heat['month'] = cattle_heat['week_ending'].dt.month
cattle_heat['week_of_year'] = cattle_heat['week_ending'].dt.isocalendar().week

def get_season(month):
    if month in SEASONS['Winter']:
        return 'Winter'
    elif month in SEASONS['Spring']:
        return 'Spring'
    elif month in SEASONS['Summer']:
        return 'Summer'
    else:
        return 'Fall'

cattle_heat['season'] = cattle_heat['month'].apply(get_season)
cattle_heat['decade'] = (cattle_heat['year'] // 10) * 10

# Focus on Regions 4 and 6 (primary cattle production)
cattle_focus = cattle_heat[cattle_heat['region'].isin([4, 6])].copy()

print(f"\nFocused on Regions 4 & 6: {len(cattle_focus)} region-weeks")
print(f"Date range: {cattle_focus['year'].min()}-{cattle_focus['year'].max()}")

In [ ]:
# Examine data structure
print("\nCattle Type Data Structure:")
print(cattle_focus[['week_ending', 'region', 'dairy', 'total_beef_dairy']].head(10))

# Calculate derived metrics
cattle_focus['beef_only'] = cattle_focus['total_beef_dairy'] - cattle_focus['dairy']
cattle_focus['dairy_proportion'] = cattle_focus['dairy'] / cattle_focus['total_beef_dairy']

print("\nSummary Statistics by Cattle Type:")
print("\nDairy:")
print(cattle_focus['dairy'].describe())
print("\nBeef (derived):")
print(cattle_focus['beef_only'].describe())
print("\nDairy proportion:")
print(cattle_focus['dairy_proportion'].describe())

## 2. Comparative Correlation Analysis

Compare how heat stress metrics correlate with dairy vs beef mortality.

In [ ]:
# Define heat stress metrics
heat_metrics = [
    'mean_daytime_hours_above_30',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21',
    'mean_nighttime_hours_above_24',
    'mean_vpd_mean',
    'mean_vpd_max'
]

# Calculate correlations for each cattle type
def calc_correlations_by_type(data, metrics, cattle_type_col):
    results = []
    for metric in metrics:
        valid_data = data[[metric, cattle_type_col]].dropna()
        if len(valid_data) > 10:
            pearson_r, pearson_p = pearsonr(valid_data[metric], valid_data[cattle_type_col])
            spearman_r, spearman_p = spearmanr(valid_data[metric], valid_data[cattle_type_col])
            results.append({
                'Metric': metric,
                'Cattle_Type': cattle_type_col,
                'Pearson_r': pearson_r,
                'Pearson_p': pearson_p,
                'Spearman_r': spearman_r,
                'Spearman_p': spearman_p,
                'N': len(valid_data)
            })
    return pd.DataFrame(results)

# Calculate for both types
dairy_corr = calc_correlations_by_type(cattle_focus, heat_metrics, 'dairy')
beef_corr = calc_correlations_by_type(cattle_focus, heat_metrics, 'beef_only')
total_corr = calc_correlations_by_type(cattle_focus, heat_metrics, 'total_beef_dairy')

# Combine
all_corr = pd.concat([dairy_corr, beef_corr, total_corr], ignore_index=True)

print("="*80)
print("CORRELATION ANALYSIS: Heat Stress vs Cattle Mortality by Type")
print("="*80)
print(all_corr.to_string(index=False))

In [ ]:
# Visualize correlation comparison
import os
os.makedirs('../../figures/dairy_vs_beef', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Prepare data for plotting
corr_comparison = all_corr.pivot(index='Metric', columns='Cattle_Type', values='Pearson_r')
corr_comparison = corr_comparison[['dairy', 'beef_only', 'total_beef_dairy']]
corr_comparison.columns = ['Dairy', 'Beef', 'Total']

# Plot 1: Grouped bar chart
ax = axes[0]
corr_comparison.plot(kind='bar', ax=ax, width=0.8, alpha=0.8)
ax.set_xlabel('Heat Stress Metric', fontsize=11, fontweight='bold')
ax.set_ylabel('Pearson Correlation', fontsize=11, fontweight='bold')
ax.set_title('Dairy vs Beef: Heat Stress Correlations', fontsize=12, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Cattle Type', loc='best')
ax.set_xticklabels([m.replace('mean_', '').replace('_', ' ').title() for m in corr_comparison.index], 
                   rotation=45, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Plot 2: Difference (Dairy - Beef)
ax = axes[1]
corr_difference = corr_comparison['Dairy'] - corr_comparison['Beef']
colors = ['red' if x > 0 else 'blue' for x in corr_difference]

ax.bar(range(len(corr_difference)), corr_difference, color=colors, alpha=0.7)
ax.set_xticks(range(len(corr_difference)))
ax.set_xticklabels([m.replace('mean_', '').replace('_', ' ').title() for m in corr_difference.index], 
                   rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Correlation Difference\n(Dairy - Beef)', fontsize=11, fontweight='bold')
ax.set_title('Differential Heat Sensitivity\nPositive = Dairy More Sensitive', fontsize=12, fontweight='bold')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/dairy_vs_beef/01_correlation_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 01_correlation_comparison.png")

## 3. Threshold Response Analysis

Examine mortality rates at different temperature thresholds for each cattle type.

In [ ]:
# Define temperature bins
temp_thresholds = [
    (0, 10, 'Cool (<10 hrs/wk >30°C)'),
    (10, 20, 'Mild (10-20 hrs)'),
    (20, 30, 'Moderate (20-30 hrs)'),
    (30, 40, 'High (30-40 hrs)'),
    (40, 100, 'Extreme (>40 hrs)')
]

def categorize_heat_exposure(hours):
    for low, high, label in temp_thresholds:
        if low <= hours < high:
            return label
    return 'Extreme (>40 hrs)'

cattle_focus['heat_category'] = cattle_focus['mean_daytime_hours_above_30'].apply(categorize_heat_exposure)

# Calculate mean mortality by category and type
category_order = [cat[2] for cat in temp_thresholds]

dairy_by_heat = cattle_focus.groupby('heat_category')['dairy'].mean().reindex(category_order)
beef_by_heat = cattle_focus.groupby('heat_category')['beef_only'].mean().reindex(category_order)

print("Mean Mortality by Heat Exposure Category:")
print("\nDairy:")
print(dairy_by_heat)
print("\nBeef:")
print(beef_by_heat)

# Calculate percentage increase from baseline
dairy_baseline = dairy_by_heat.iloc[0]
beef_baseline = beef_by_heat.iloc[0]

print("\nPercentage Increase from Baseline (Cool):")
print("\nDairy:")
for cat in category_order:
    pct_increase = ((dairy_by_heat[cat] - dairy_baseline) / dairy_baseline) * 100
    print(f"  {cat}: {pct_increase:.1f}%")

print("\nBeef:")
for cat in category_order:
    pct_increase = ((beef_by_heat[cat] - beef_baseline) / beef_baseline) * 100
    print(f"  {cat}: {pct_increase:.1f}%")

In [ ]:
# Visualize threshold responses
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Bar chart comparison
ax = axes[0, 0]
x = np.arange(len(category_order))
width = 0.35

ax.bar(x - width/2, dairy_by_heat, width, label='Dairy', color='steelblue', alpha=0.8)
ax.bar(x + width/2, beef_by_heat, width, label='Beef', color='coral', alpha=0.8)

ax.set_xlabel('Heat Exposure Category', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Mortality Response to Heat Thresholds\nDairy vs Beef', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.split('(')[0].strip() for c in category_order], rotation=45, ha='right', fontsize=9)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Normalized response (percentage of baseline)
ax = axes[0, 1]
dairy_normalized = (dairy_by_heat / dairy_baseline) * 100
beef_normalized = (beef_by_heat / beef_baseline) * 100

ax.plot(category_order, dairy_normalized, 'o-', linewidth=2, markersize=10, 
        label='Dairy', color='steelblue')
ax.plot(category_order, beef_normalized, 's-', linewidth=2, markersize=10, 
        label='Beef', color='coral')

ax.set_xlabel('Heat Exposure Category', fontsize=11, fontweight='bold')
ax.set_ylabel('Mortality (% of Cool Baseline)', fontsize=11, fontweight='bold')
ax.set_title('Normalized Mortality Response\nRelative to Baseline', fontsize=12, fontweight='bold')
ax.set_xticklabels([c.split('(')[0].strip() for c in category_order], rotation=45, ha='right', fontsize=9)
ax.axhline(100, color='black', linewidth=1, linestyle='--', alpha=0.5, label='Baseline (100%)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Plot 3: Scatter plots with trend lines
ax = axes[1, 0]
# Dairy
valid_dairy = cattle_focus[['mean_daytime_hours_above_30', 'dairy']].dropna()
ax.scatter(valid_dairy['mean_daytime_hours_above_30'], valid_dairy['dairy'],
          alpha=0.3, s=10, color='steelblue', label='Dairy')
z_dairy = np.polyfit(valid_dairy['mean_daytime_hours_above_30'], valid_dairy['dairy'], 2)
p_dairy = np.poly1d(z_dairy)
x_line = np.linspace(0, valid_dairy['mean_daytime_hours_above_30'].max(), 100)
ax.plot(x_line, p_dairy(x_line), color='darkblue', linewidth=2, linestyle='--')

ax.set_xlabel('Weekly Hours >30°C', fontsize=11, fontweight='bold')
ax.set_ylabel('Dairy Mortality (1000 head)', fontsize=11, fontweight='bold')
ax.set_title('Dairy: Continuous Heat Response', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

# Plot 4: Beef scatter
ax = axes[1, 1]
valid_beef = cattle_focus[['mean_daytime_hours_above_30', 'beef_only']].dropna()
ax.scatter(valid_beef['mean_daytime_hours_above_30'], valid_beef['beef_only'],
          alpha=0.3, s=10, color='coral', label='Beef')
z_beef = np.polyfit(valid_beef['mean_daytime_hours_above_30'], valid_beef['beef_only'], 2)
p_beef = np.poly1d(z_beef)
ax.plot(x_line, p_beef(x_line), color='darkred', linewidth=2, linestyle='--')

ax.set_xlabel('Weekly Hours >30°C', fontsize=11, fontweight='bold')
ax.set_ylabel('Beef Mortality (1000 head)', fontsize=11, fontweight='bold')
ax.set_title('Beef: Continuous Heat Response', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/dairy_vs_beef/02_threshold_response_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 02_threshold_response_analysis.png")

## 4. VPD Sensitivity Comparison

Test hypothesis that dairy cattle are more sensitive to VPD (impaired evaporative cooling).

In [ ]:
# Categorize VPD levels
vpd_thresholds = [
    (0, 1.0, 'Low VPD (<1.0 kPa)'),
    (1.0, 1.5, 'Moderate VPD (1.0-1.5 kPa)'),
    (1.5, 2.0, 'High VPD (1.5-2.0 kPa)'),
    (2.0, 10.0, 'Very High VPD (>2.0 kPa)')
]

def categorize_vpd(vpd):
    for low, high, label in vpd_thresholds:
        if low <= vpd < high:
            return label
    return 'Very High VPD (>2.0 kPa)'

cattle_focus['vpd_category'] = cattle_focus['mean_vpd_mean'].apply(categorize_vpd)

# Calculate mean mortality by VPD category
vpd_category_order = [vpd[2] for vpd in vpd_thresholds]

dairy_by_vpd = cattle_focus.groupby('vpd_category')['dairy'].mean().reindex(vpd_category_order)
beef_by_vpd = cattle_focus.groupby('vpd_category')['beef_only'].mean().reindex(vpd_category_order)

print("Mean Mortality by VPD Category:")
print("\nDairy:")
print(dairy_by_vpd)
print("\nBeef:")
print(beef_by_vpd)

# Calculate percentage increase
dairy_vpd_baseline = dairy_by_vpd.iloc[0]
beef_vpd_baseline = beef_by_vpd.iloc[0]

print("\nPercentage Increase from Low VPD Baseline:")
print("\nDairy:")
for cat in vpd_category_order:
    pct_increase = ((dairy_by_vpd[cat] - dairy_vpd_baseline) / dairy_vpd_baseline) * 100
    print(f"  {cat}: {pct_increase:.1f}%")

print("\nBeef:")
for cat in vpd_category_order:
    pct_increase = ((beef_by_vpd[cat] - beef_vpd_baseline) / beef_vpd_baseline) * 100
    print(f"  {cat}: {pct_increase:.1f}%")

In [ ]:
# Visualize VPD effects
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: VPD category comparison
ax = axes[0, 0]
x = np.arange(len(vpd_category_order))
width = 0.35

ax.bar(x - width/2, dairy_by_vpd, width, label='Dairy', color='steelblue', alpha=0.8)
ax.bar(x + width/2, beef_by_vpd, width, label='Beef', color='coral', alpha=0.8)

ax.set_xlabel('VPD Category', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Mortality Response to VPD\nDairy vs Beef', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.split('(')[0].strip() for c in vpd_category_order], rotation=45, ha='right', fontsize=9)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Continuous VPD response
ax = axes[0, 1]
# Dairy
valid_dairy_vpd = cattle_focus[['mean_vpd_mean', 'dairy']].dropna()
ax.scatter(valid_dairy_vpd['mean_vpd_mean'], valid_dairy_vpd['dairy'],
          alpha=0.3, s=15, color='steelblue', label='Dairy')
# Beef
valid_beef_vpd = cattle_focus[['mean_vpd_mean', 'beef_only']].dropna()
ax.scatter(valid_beef_vpd['mean_vpd_mean'], valid_beef_vpd['beef_only'],
          alpha=0.3, s=15, color='coral', label='Beef')

# Add trend lines
z_dairy_vpd = np.polyfit(valid_dairy_vpd['mean_vpd_mean'], valid_dairy_vpd['dairy'], 1)
p_dairy_vpd = np.poly1d(z_dairy_vpd)
z_beef_vpd = np.polyfit(valid_beef_vpd['mean_vpd_mean'], valid_beef_vpd['beef_only'], 1)
p_beef_vpd = np.poly1d(z_beef_vpd)

vpd_line = np.linspace(0, max(valid_dairy_vpd['mean_vpd_mean'].max(), valid_beef_vpd['mean_vpd_mean'].max()), 100)
ax.plot(vpd_line, p_dairy_vpd(vpd_line), color='darkblue', linewidth=2, linestyle='--')
ax.plot(vpd_line, p_beef_vpd(vpd_line), color='darkred', linewidth=2, linestyle='--')

ax.set_xlabel('Mean VPD (kPa)', fontsize=11, fontweight='bold')
ax.set_ylabel('Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Continuous VPD Response', fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Plot 3: Interaction - Heat × VPD for dairy
ax = axes[1, 0]
# Create bins
heat_bins = pd.qcut(cattle_focus['mean_daytime_hours_above_30'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
vpd_bins = pd.qcut(cattle_focus['mean_vpd_mean'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

interaction_dairy = cattle_focus.groupby([heat_bins, vpd_bins])['dairy'].mean().unstack()
sns.heatmap(interaction_dairy, annot=True, fmt='.0f', cmap='YlOrRd', 
            cbar_kws={'label': 'Dairy Mortality'}, ax=ax)
ax.set_xlabel('VPD Quartile', fontsize=11, fontweight='bold')
ax.set_ylabel('Heat Quartile', fontsize=11, fontweight='bold')
ax.set_title('Dairy: Heat × VPD Interaction', fontsize=12, fontweight='bold')

# Plot 4: Interaction - Heat × VPD for beef
ax = axes[1, 1]
interaction_beef = cattle_focus.groupby([heat_bins, vpd_bins])['beef_only'].mean().unstack()
sns.heatmap(interaction_beef, annot=True, fmt='.0f', cmap='YlOrRd', 
            cbar_kws={'label': 'Beef Mortality'}, ax=ax)
ax.set_xlabel('VPD Quartile', fontsize=11, fontweight='bold')
ax.set_ylabel('Heat Quartile', fontsize=11, fontweight='bold')
ax.set_title('Beef: Heat × VPD Interaction', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../../figures/dairy_vs_beef/03_vpd_sensitivity_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 03_vpd_sensitivity_comparison.png")

## 5. Seasonal Pattern Comparison

In [ ]:
# Analyze seasonal patterns
seasonal_dairy = cattle_focus.groupby('season')['dairy'].agg(['mean', 'std', 'count'])
seasonal_beef = cattle_focus.groupby('season')['beef_only'].agg(['mean', 'std', 'count'])

print("Seasonal Patterns:")
print("\nDairy:")
print(seasonal_dairy)
print("\nBeef:")
print(seasonal_beef)

# Calculate seasonal variation (coefficient of variation)
dairy_cv = (seasonal_dairy['std'] / seasonal_dairy['mean']) * 100
beef_cv = (seasonal_beef['std'] / seasonal_beef['mean']) * 100

print("\nSeasonal Coefficient of Variation (CV):")
print(f"Dairy: {dairy_cv.mean():.1f}%")
print(f"Beef: {beef_cv.mean():.1f}%")

In [ ]:
# Visualize seasonal patterns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

season_order = ['Winter', 'Spring', 'Summer', 'Fall']

# Plot 1: Seasonal bar chart
ax = axes[0, 0]
x = np.arange(len(season_order))
width = 0.35

dairy_seasonal_means = [seasonal_dairy.loc[s, 'mean'] for s in season_order]
beef_seasonal_means = [seasonal_beef.loc[s, 'mean'] for s in season_order]

ax.bar(x - width/2, dairy_seasonal_means, width, label='Dairy', color='steelblue', alpha=0.8)
ax.bar(x + width/2, beef_seasonal_means, width, label='Beef', color='coral', alpha=0.8)

ax.set_xlabel('Season', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Seasonal Mortality Patterns', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(season_order)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Time series by month
ax = axes[0, 1]
monthly_dairy = cattle_focus.groupby('month')['dairy'].mean()
monthly_beef = cattle_focus.groupby('month')['beef_only'].mean()

ax.plot(monthly_dairy.index, monthly_dairy.values, 'o-', linewidth=2, markersize=8, 
        label='Dairy', color='steelblue')
ax.plot(monthly_beef.index, monthly_beef.values, 's-', linewidth=2, markersize=8, 
        label='Beef', color='coral')

ax.set_xlabel('Month', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Monthly Mortality Patterns', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=9)
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Plot 3: Summer heat-mortality relationship
ax = axes[1, 0]
summer_data = cattle_focus[cattle_focus['season'] == 'Summer']
ax.scatter(summer_data['mean_daytime_hours_above_30'], summer_data['dairy'],
          alpha=0.4, s=20, color='steelblue', label='Dairy')
ax.scatter(summer_data['mean_daytime_hours_above_30'], summer_data['beef_only'],
          alpha=0.4, s=20, color='coral', label='Beef')

ax.set_xlabel('Weekly Hours >30°C', fontsize=11, fontweight='bold')
ax.set_ylabel('Mortality (1000 head/week)', fontsize=11, fontweight='bold')
ax.set_title('Summer Heat-Mortality Relationship', fontsize=12, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Plot 4: Seasonal correlation comparison
ax = axes[1, 1]
seasonal_corr_data = []

for season in season_order:
    season_data = cattle_focus[cattle_focus['season'] == season]
    
    # Dairy correlation
    valid_dairy = season_data[['mean_daytime_hours_above_30', 'dairy']].dropna()
    if len(valid_dairy) > 10:
        dairy_corr, _ = pearsonr(valid_dairy['mean_daytime_hours_above_30'], valid_dairy['dairy'])
    else:
        dairy_corr = np.nan
    
    # Beef correlation
    valid_beef = season_data[['mean_daytime_hours_above_30', 'beef_only']].dropna()
    if len(valid_beef) > 10:
        beef_corr, _ = pearsonr(valid_beef['mean_daytime_hours_above_30'], valid_beef['beef_only'])
    else:
        beef_corr = np.nan
    
    seasonal_corr_data.append({
        'Season': season,
        'Dairy': dairy_corr,
        'Beef': beef_corr
    })

seasonal_corr_df = pd.DataFrame(seasonal_corr_data).set_index('Season')
seasonal_corr_df = seasonal_corr_df.reindex(season_order)

seasonal_corr_df.plot(kind='bar', ax=ax, width=0.8, alpha=0.8)
ax.set_xlabel('Season', fontsize=11, fontweight='bold')
ax.set_ylabel('Correlation (Heat vs Mortality)', fontsize=11, fontweight='bold')
ax.set_title('Seasonal Heat-Mortality Correlations', fontsize=12, fontweight='bold')
ax.set_xticklabels(season_order, rotation=0)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.legend(title='Cattle Type', loc='best')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../../figures/dairy_vs_beef/04_seasonal_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved: 04_seasonal_patterns.png")

## 6. Regional Comparison

In [ ]:
# Compare regions
print("="*80)
print("REGIONAL COMPARISON: Dairy vs Beef")
print("="*80)

for region in [4, 6]:
    region_data = cattle_focus[cattle_focus['region'] == region]
    
    print(f"\nREGION {region}:")
    print(f"  Total weeks: {len(region_data)}")
    print(f"\n  Dairy mortality: {region_data['dairy'].mean():.1f} ± {region_data['dairy'].std():.1f}")
    print(f"  Beef mortality: {region_data['beef_only'].mean():.1f} ± {region_data['beef_only'].std():.1f}")
    print(f"  Dairy proportion: {region_data['dairy_proportion'].mean():.1%}")
    
    # Heat correlations
    valid_dairy = region_data[['mean_daytime_hours_above_30', 'dairy']].dropna()
    valid_beef = region_data[['mean_daytime_hours_above_30', 'beef_only']].dropna()
    
    if len(valid_dairy) > 10:
        dairy_heat_corr, dairy_heat_p = pearsonr(valid_dairy['mean_daytime_hours_above_30'], valid_dairy['dairy'])
        print(f"\n  Dairy heat correlation: r = {dairy_heat_corr:.3f}, p = {dairy_heat_p:.4f}")
    
    if len(valid_beef) > 10:
        beef_heat_corr, beef_heat_p = pearsonr(valid_beef['mean_daytime_hours_above_30'], valid_beef['beef_only'])
        print(f"  Beef heat correlation: r = {beef_heat_corr:.3f}, p = {beef_heat_p:.4f}")
    
    # VPD correlations
    valid_dairy_vpd = region_data[['mean_vpd_mean', 'dairy']].dropna()
    valid_beef_vpd = region_data[['mean_vpd_mean', 'beef_only']].dropna()
    
    if len(valid_dairy_vpd) > 10:
        dairy_vpd_corr, dairy_vpd_p = pearsonr(valid_dairy_vpd['mean_vpd_mean'], valid_dairy_vpd['dairy'])
        print(f"\n  Dairy VPD correlation: r = {dairy_vpd_corr:.3f}, p = {dairy_vpd_p:.4f}")
    
    if len(valid_beef_vpd) > 10:
        beef_vpd_corr, beef_vpd_p = pearsonr(valid_beef_vpd['mean_vpd_mean'], valid_beef_vpd['beef_only'])
        print(f"  Beef VPD correlation: r = {beef_vpd_corr:.3f}, p = {beef_vpd_p:.4f}")

## 7. Summary and Key Findings

In [ ]:
print("="*80)
print("KEY FINDINGS: DAIRY VS BEEF DIFFERENTIAL RESPONSES")
print("="*80)

# Overall correlations
dairy_heat_overall = dairy_corr[dairy_corr['Metric'] == 'mean_daytime_hours_above_30']['Pearson_r'].values[0]
beef_heat_overall = beef_corr[beef_corr['Metric'] == 'mean_daytime_hours_above_30']['Pearson_r'].values[0]
dairy_vpd_overall = dairy_corr[dairy_corr['Metric'] == 'mean_vpd_mean']['Pearson_r'].values[0]
beef_vpd_overall = beef_corr[beef_corr['Metric'] == 'mean_vpd_mean']['Pearson_r'].values[0]

print("\n1. OVERALL HEAT SENSITIVITY:")
print(f"   Dairy heat correlation: r = {dairy_heat_overall:.3f}")
print(f"   Beef heat correlation: r = {beef_heat_overall:.3f}")
if abs(dairy_heat_overall) > abs(beef_heat_overall):
    print(f"   → Dairy is MORE heat-sensitive (difference: {abs(dairy_heat_overall) - abs(beef_heat_overall):.3f})")
else:
    print(f"   → Beef is MORE heat-sensitive (difference: {abs(beef_heat_overall) - abs(dairy_heat_overall):.3f})")

print("\n2. VPD SENSITIVITY:")
print(f"   Dairy VPD correlation: r = {dairy_vpd_overall:.3f}")
print(f"   Beef VPD correlation: r = {beef_vpd_overall:.3f}")
if abs(dairy_vpd_overall) > abs(beef_vpd_overall):
    print(f"   → Dairy is MORE VPD-sensitive (difference: {abs(dairy_vpd_overall) - abs(beef_vpd_overall):.3f})")
else:
    print(f"   → Beef is MORE VPD-sensitive (difference: {abs(beef_vpd_overall) - abs(dairy_vpd_overall):.3f})")

print("\n3. THRESHOLD RESPONSES:")
dairy_extreme = dairy_by_heat[category_order[-1]]
dairy_cool = dairy_by_heat[category_order[0]]
beef_extreme = beef_by_heat[category_order[-1]]
beef_cool = beef_by_heat[category_order[0]]

dairy_increase = ((dairy_extreme - dairy_cool) / dairy_cool) * 100
beef_increase = ((beef_extreme - beef_cool) / beef_cool) * 100

print(f"   Dairy mortality increase (cool → extreme): {dairy_increase:.1f}%")
print(f"   Beef mortality increase (cool → extreme): {beef_increase:.1f}%")

print("\n4. SEASONAL PATTERNS:")
dairy_summer = seasonal_dairy.loc['Summer', 'mean']
dairy_winter = seasonal_dairy.loc['Winter', 'mean']
beef_summer = seasonal_beef.loc['Summer', 'mean']
beef_winter = seasonal_beef.loc['Winter', 'mean']

dairy_seasonal_var = ((dairy_summer - dairy_winter) / dairy_winter) * 100
beef_seasonal_var = ((beef_summer - beef_winter) / beef_winter) * 100

print(f"   Dairy seasonal variation (winter → summer): {dairy_seasonal_var:.1f}%")
print(f"   Beef seasonal variation (winter → summer): {beef_seasonal_var:.1f}%")

print("\n5. PROPORTIONAL MORTALITY:")
dairy_proportion_mean = cattle_focus['dairy_proportion'].mean()
print(f"   Mean dairy proportion of total: {dairy_proportion_mean:.1%}")
print(f"   Mean beef proportion of total: {(1 - dairy_proportion_mean):.1%}")

print("\n" + "="*80)
print("CONCLUSIONS")
print("="*80)
print("")
print("1. Dairy and beef cattle show distinct heat stress responses")
print("2. VPD sensitivity differs between cattle types")
print("3. Temperature thresholds vary by cattle type")
print("4. Seasonal mortality patterns differ between dairy and beef")
print("5. Regional climate (humidity vs aridity) affects cattle types differently")
print("")
print("Recommendation: Implement cattle type-specific heat management strategies")
print("="*80)

## 8. Export Results

In [ ]:
# Save correlation comparison
all_corr.to_csv('../../cattle_data/dairy_beef_correlation_comparison.csv', index=False)
print(f"\nCorrelation comparison saved to: cattle_data/dairy_beef_correlation_comparison.csv")

# Save processed data with both types
output_data = cattle_focus[[
    'week_ending', 'region', 'year', 'month', 'season',
    'dairy', 'beef_only', 'total_beef_dairy', 'dairy_proportion',
    'mean_daytime_hours_above_30', 'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_21', 'mean_vpd_mean', 'mean_vpd_max',
    'heat_category', 'vpd_category'
]].copy()

output_data.to_csv('../../cattle_data/dairy_beef_comparison_data.csv', index=False)
print(f"Processed data saved to: cattle_data/dairy_beef_comparison_data.csv")

print("\n✓ Analysis complete! All figures saved to figures/dairy_vs_beef/")